<h1>Step 1 : Import Libraries</h1>

In [ ]:
# keep the output clean: the most extreme C values we will try never fully
# converge (and never win) — this line stops those warnings cluttering the screen
import os
os.environ["PYTHONWARNINGS"] = "ignore::UserWarning"

# numpy            -> the range of C values we will try
# linear_model     -> Logistic Regression
# decomposition    -> PCA
# datasets         -> sklearn's built-in datasets (includes the wine data)
# Pipeline         -> chains scaling, PCA and the model into one unit
# GridSearchCV     -> tries every parameter combination and keeps the best
# cross_val_score  -> scores the finished pipeline fairly
# StandardScaler   -> puts all features on the same scale
import numpy as np
from sklearn import linear_model, decomposition, datasets
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler

<h1>Step 2 : Import Dataset</h1>

In [ ]:
# load the wine dataset that ships inside sklearn - same 178 Italian wines
# as the previous two labs: 13 measurements (X) and the wine type (y)
wines = datasets.load_wine()
X = wines.data
y = wines.target



<h1>Step 3 : Normalise and Feature Extraction</h1>
In this step we will standardise the features, and then decompose the dataset to determine the best number of features to include without losing the variance. In the last line you will then create an instance of LogisticRegression. 

In [ ]:
# the three building blocks of our pipeline
std_scaler = StandardScaler()      # scale the 13 measurements
pca = decomposition.PCA()          # reduce the dimensions (how many? the grid search decides)

# solver='liblinear' supports BOTH penalties we want to tune (l1 and l2);
# max_iter=2000 gives it room to converge across the whole range of C values
logit = linear_model.LogisticRegression(solver='liblinear', max_iter=2000)

<h1>Step 4 : Create the pipeline</h1>
Pipeline will pass the modules one by one through GridSearchCV for which we want to get the best parameters. Hence making an object pipe to create a pipeline for std_scaler,pca and logit.

In [ ]:
# chain the three steps into one pipeline: every piece of data flows
# through scaling -> PCA -> logistic regression, in that order
pipe = Pipeline(steps=[('std_scaler', std_scaler),
                           ('pca', pca),
                           ('logit', logit)])

<h1>Step 5 : Range for PCA</h1>
This step creates the range for PCA as we want to optimise the number of components using PCA


In [ ]:
# candidate values for the number of PCA components: 1 to 13
n_components = list(range(1, X.shape[1] + 1, 1))

<h1>Step 6 : Create Parameters for Logistic Regression</h1>
Logistic Regression will require two parameters 'C' and 'penalty' to be optimised by GridSearchCV. the following code will do that.

In [ ]:
# candidate values for the two Logistic Regression parameters:
# C       -> regularisation strength: 50 values from 0.0001 to 10000
# penalty -> the type of regularisation, l1 or l2
C = np.logspace(-4, 4, 50)
penalty = ['l1', 'l2']

<h1>Step 7 : Dictionary to set all the parameters</h1>


In [ ]:
# gather every candidate into one dictionary for the grid search
# the names ('pca__', 'logit__') tell it which pipeline step each belongs to
parameters = dict(pca__n_components=n_components,
                      logit__C=C,
                      logit__penalty=penalty)

<h1>Step 8 : Apply GridSearchCV </h1>
Some important parameters to consider for GridSearchCV:
<ul>
<li> estimator: used to pass the models or functions on which we want to use GridSearchCV </li>
<li> param_grid: It can be a list or dictionary of parameters of models or function in which GridSearchCV have to select the best.</li>
<li>Scoring: It specifies the evaluating metric for the model performance to decide the best hyperparameters,uses estimator score by default </li>
</ul>

In [ ]:
# try EVERY combination: 13 components x 50 C values x 2 penalties = 1,300
# combinations, each tested with 5-fold cross-validation = 6,500 model fits.
# n_jobs=-1 uses every CPU core - even so, expect about a minute.
clf = GridSearchCV(pipe, parameters, n_jobs=-1)
clf.fit(X, y)

<h1>Step 9 : Printing the results</h1>

In [ ]:
# the winning combination found by the grid search
print('Best Penalty:', clf.best_estimator_.get_params()['logit__penalty'])
print('Best C:', clf.best_estimator_.get_params()['logit__C'])
print('Best Number Of Components:', clf.best_estimator_.get_params()['pca__n_components'])
print()
print(clf.best_estimator_.get_params()['logit'])

# final check: cross-validate the whole tuned pipeline on 8 fresh splits
# (this re-runs the entire search 8 times - expect a few minutes)
CV_Results = cross_val_score(clf, X, y, cv=8, n_jobs= -1)
print(CV_Results)
print(CV_Results.mean())
print(CV_Results.std())


<h1>What did we learn?</h1>

- **A pipeline chains the steps into one unit.** Scaling, PCA and the model became a
  single object, so every candidate was tested with the same, correctly ordered recipe.
- **Grid search automates the tuning.** Instead of guessing the settings, we let
  GridSearchCV try all 1,300 combinations (6,500 model fits) and keep the best.
- **The search made real decisions for us:** l2 penalty, C ≈ 1.2, and 12 principal
  components out of 13 — choices backed by cross-validated evidence, not intuition.
- **Cross-validation gives the honest final score.** The tuned pipeline averaged about
  97% across 8 fresh splits — that, not any single lucky split, is the number to report.
- **This is how the pieces fit together.** Scaling (Week 3), PCA (this week) and
  classification joined into one tuned, tested workflow — the standard shape of a
  real machine-learning project.